In [0]:
%pip install azure-eventhub nest-asyncio

import asyncio, random
from datetime import datetime, timedelta, timezone
from azure.eventhub import EventData
from azure.eventhub.aio import EventHubProducerClient
from sensor_stream import make_event, SITES

EH_NAME     = "gabrielajaniszews786_eventhub"
EH_CONN_STR = dbutils.secrets.get("default2", "eventhub-con-str-gabriela")

N_DAYS = 45   # how many days back to seed

def build_events():
    """Yield one backdated MeterEvent per (day, hour, site) over the last N_DAYS.
    timestamp_utc is aligned to the hour so it joins cleanly to hourly ENTSO-E prices."""
    now = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
    for day_offset in range(N_DAYS, 0, -1):          # oldest -> newest
        for hour in range(24):
            ts = (now - timedelta(days=day_offset)).replace(hour=hour)
            ts_iso = ts.strftime("%Y-%m-%dT%H:%M:%SZ")
            for site in SITES:
                avg_power = random.uniform(700.0, 800.0)
                yield make_event(
                    site,
                    timestamp_utc=ts_iso,
                    consumption_kwh=avg_power * random.uniform(0.85, 1.25),  # correlated with power
                    avg_power_kw=avg_power,
                    pue=random.uniform(1.2, 1.4),
                )

async def run():
    producer = EventHubProducerClient.from_connection_string(
        conn_str=EH_CONN_STR, eventhub_name=EH_NAME)
    sent = 0
    async with producer:
        batch = await producer.create_batch()
        for event in build_events():
            try:
                batch.add(EventData(event.to_json_bytes()))
            except ValueError:                       # batch full -> flush, start a new one
                await producer.send_batch(batch)
                sent += len(batch)
                batch = await producer.create_batch()
                batch.add(EventData(event.to_json_bytes()))
        if len(batch) > 0:                           # flush the final partial batch
            await producer.send_batch(batch)
            sent += len(batch)
    print(f"Backfill done: sent {sent} events over {N_DAYS} days x 24h x {len(SITES)} sites")

try:
    asyncio.run(run())
except RuntimeError:                                 # already in a running loop (Databricks)
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.run(run())